In [51]:
import pandas as pd
import xarray as xr
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import glob

In [52]:
# 1. Pad naar jouw Parquet-bestand
# BASE_DIR = "/Users/gjwdijk/kedro/wf-kedro/data/01_raw/parquet/heart_rate_sample/"
BASE_DIR= f'/Volumes/Extreme SSD/pq/parquet/heart_rate_sample/'
#profile_id=9
output_path = f"{BASE_DIR}profile_id={profile_id}/"
output_path = f"{BASE_DIR}"

In [54]:
# 2. Lees het Parquet-bestand in met Pandas
# We gaan ervan uit dat er kolommen zijn voor de tijd en eventuele andere dimensies/variabelen
# Scan de gehele map als een dataset
# dataset = ds.dataset(output_path, format="parquet")
# dataset
files = glob.glob(f"{BASE_DIR}/**/*.parquet", recursive=True)

dfs = []
for f in files:
    try:
        # Lees het bestand individueel in
        df_part = pq.read_table(f).to_pandas()
        dfs.append(df_part)
    except Exception as e:
        print(f"Fout bij bestand {f}: {e}")
df = pd.concat(dfs, ignore_index=True)
len(df), df.head()


(
    10267490,
                     time            source duration  \
0 2023-05-22 22:00:00  weconnect.garmin    PT15M   
1 2023-05-22 22:15:00  weconnect.garmin    PT15M   
2 2023-05-22 22:30:00  weconnect.garmin    PT15M   
3 2023-05-22 22:45:00  weconnect.garmin    PT15M   
4 2023-05-22 23:00:00  weconnect.garmin    PT15M   

                                  heart_rate_samples  \
0  [[15, 71], [30, 71], [45, 71], [60, 71], [75, ...   
1  [[0, 67], [15, 68], [30, 68], [45, 68], [60, 6...   
2  [[0, 66], [15, 65], [30, 65], [45, 65], [60, 6...   
3  [[0, 65], [15, 62], [30, 62], [45, 62], [60, 6...   
4  [[0, 64], [15, 64], [30, 64], [45, 64], [60, 6...   

                                                  id  is_manual  max  min  \
0  b'rM\xc4\xc4&\x83K\xbf\xbd\xeda\x9d\x08\x8d\x0...        NaN   72   67   
1           b'\x84O\x8bg(\xfdI\xa2\x98\xe8;9\x93Sr1'        NaN   68   66   
2     b'm\x02>!\x81\xa2F\x99\xa6g\xcbIb\xdf\xcd\x1d'        NaN   67   64   
3        b'5n/\x82H;L

In [61]:
ts = df.iloc[400]
type(ts)
ts.time, ts.profile_id, ts.heart_rate_samples


(
    Timestamp('2023-05-27 03:30:00'),
    np.int32(10092816),
    array([array([ 0, 60]), array([15, 62]), array([30, 62]), array([45, 62]),
       array([60, 62]), array([75, 76]), array([90, 76]),
       array([105,  76]), array([120,  76]), array([135,  72]),
       array([150,  72]), array([165,  72]), array([180,  72]),
       array([195,  57]), array([210,  57]), array([225,  57]),
       array([240,  57]), array([255,  55]), array([270,  55]),
       array([285,  55]), array([300,  55]), array([315,  55]),
       array([330,  55]), array([345,  55]), array([360,  55]),
       array([375,  54]), array([390,  54]), array([405,  54]),
       array([420,  54]), array([435,  59]), array([450,  59]),
       array([465,  59]), array([480,  59]), array([495,  73]),
       array([510,  73]), array([525,  73]), array([540,  73]),
       array([555,  74]), array([570,  74]), array([585,  74]),
       array([600,  74]), array([615,  82]), array([630,  82]),
       array([645,  82]), arra

In [ ]:
# 3. Zorg dat de tijdskolom het juiste datetime-formaat heeft
# Pas 'datum_tijd_kolom' aan naar de exacte naam van jouw tijdsas
df['datum_tijd_kolom'] = pd.to_datetime(df['datum_tijd_kolom'])

# 4. Zet de kolommen die de 'dimensies' vormen als index van de DataFrame.
# Voor een simpele tijdreeks is dit alleen de tijdsas.
# Als je ook locaties hebt (bijv. sensoren of coördinaten), voeg je die toe: ['datum_tijd_kolom', 'sensor_id']
df = df.set_index(['datum_tijd_kolom'])

# 5. Converteer de Pandas DataFrame naar een Xarray Dataset
ds = xr.Dataset.from_dataframe(df)

# 6. Controleer het resultaat
print(ds)
